# EDA detallado de los datos de precipitación

Este notebook explora los archivos disponibles para el proyecto del Valle del Cauca. La meta es entender qué representa cada fuente y detectar problemas de calidad **sin corregir ni eliminar registros automáticamente**. Las alertas son hipótesis que deben revisarse con los metadatos de estaciones y del producto satelital.

## Preguntas que vamos a responder
1. ¿Qué archivos hay y qué periodos cubren?
2. ¿Hay nulos explícitos, fechas inválidas, duplicados o huecos implícitos?
3. ¿Qué significan los ceros y qué valores requieren revisión?
4. ¿Qué tan completos son los años por estación?
5. ¿Qué puede y qué no puede compararse entre estaciones y satélite?

> Ejecuta las celdas en orden. Se necesitan `pandas`, `numpy`, `matplotlib` y `openpyxl`. `rasterio` es opcional para leer metadatos de los TIFF. El notebook no cambia los archivos de entrada.

## 1. Preparación y localización de los archivos

El Excel es grande; leerlo puede tardar. El bloque busca los datos junto al notebook o en la carpeta de trabajo del proyecto.

In [ ]:
from pathlib import Path
import zipfile
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 30)

# Si el notebook está en otra carpeta, cambia esta ruta a la del proyecto.
ROOT = Path.cwd()
required = ['P_1980_1990.csv', 'P_1991_2003.csv', 'P_1930_1979.xlsx', 'ImagenesSatelitePentadal.zip']
if not all((ROOT / f).exists() for f in required):
    candidates = [Path('/Users/usuario/Documents/gustavo/Analitica/proyecto3'), Path.cwd()]
    ROOT = next((p for p in candidates if all((p / f).exists() for f in required)), ROOT)
print('Carpeta de datos:', ROOT)
print('Archivos encontrados:')
for f in required:
    p = ROOT / f
    print(f' - {f}: {p.stat().st_size / 1e6:.2f} MB' if p.exists() else f' - FALTA: {f}')

Carpeta de datos: /content
Archivos encontrados:
 - FALTA: P_1980_1990.csv
 - FALTA: P_1991_2003.csv
 - FALTA: P_1930_1979.xlsx
 - FALTA: ImagenesSatelitePentadal.zip


## 2. Carga y forma de las tablas de estaciones

`VALUE` es el valor de precipitación registrado, pero los archivos no traen aquí una unidad visible para los CSV. No asumimos milímetros hasta confirmarlo con la documentación. El Excel incluye una columna `UNITS` que revisaremos aparte.

In [5]:
csv_files = {
    'csv_1980_1990': ROOT / 'P_1980_1990.csv',
    'csv_1991_2003': ROOT / 'P_1991_2003.csv',
}
frames = {}
for source, path in csv_files.items():
    frames[source] = pd.read_csv(path, dtype={'STATION_ID': 'string'})
    print('\n', source, frames[source].shape)
    display(frames[source].head())

# El histórico se carga por separado para no mezclar periodos ni fuentes.
hist = pd.read_excel(ROOT / 'P_1930_1979.xlsx', sheet_name='Hoja1', dtype={'STATION_ID': 'string'})
print('Excel histórico:', hist.shape)
display(hist.head())
print('Columnas CSV:', list(frames['csv_1980_1990'].columns))
print('Columnas Excel:', list(hist.columns))
if 'UNITS' in hist.columns:
    display(hist['UNITS'].value_counts(dropna=False))

FileNotFoundError: [Errno 2] No such file or directory: '/content/P_1980_1990.csv'

## 3. Nulos explícitos y problemas de formato

Un nulo explícito es una celda vacía o un marcador que pandas reconoce como nulo. Esto es distinto a una fecha que no aparece en absoluto. Revisamos además si fechas o mediciones no se pueden convertir a números/fechas válidos.

In [ ]:
def null_audit(df, name):
    out = pd.DataFrame({
        "columna": df.columns,
        "nulos": [int(df[c].isna().sum()) for c in df.columns],
        "porcentaje_nulos": [float(df[c].isna().mean() * 100) for c in df.columns],
        "tipo": [str(df[c].dtype) for c in df.columns],
    })
    out.insert(0, "fuente", name)
    return out

nulls = pd.concat([null_audit(df, name) for name, df in frames.items()] + [null_audit(hist, "excel_1930_1979")], ignore_index=True)
display(nulls)

def prepare_station(df, source):
    x = df.copy()
    x["source"] = source
    for col in ["YEAR", "MONTH", "DAY", "VALUE"]:
        x[col + "_num"] = pd.to_numeric(x[col], errors="coerce")
    x["DATE"] = pd.to_datetime(dict(year=x.YEAR_num, month=x.MONTH_num, day=x.DAY_num), errors="coerce")
    return x

station = pd.concat([prepare_station(df, name) for name, df in frames.items()], ignore_index=True)
history = prepare_station(hist, "excel_1930_1979")
format_audit = pd.DataFrame([
    {"fuente": name, "filas": len(df), "id_nulo": int(df.STATION_ID.isna().sum()),
     "valor_no_numerico": int(pd.to_numeric(df.VALUE, errors="coerce").isna().sum()),
     "fecha_invalida": int(prepare_station(df, name).DATE.isna().sum())}
    for name, df in list(frames.items()) + [("excel_1930_1979", hist)]
])
display(format_audit)


## 4. ¿Qué tan completos son los registros diarios?

Para cada estación-año contamos filas, fechas únicas y cobertura del calendario. Los días faltantes se calculan como días del año menos fechas únicas observadas. Esto identifica huecos implícitos; no determina si la falta es accidental ni si un cero es dato faltante.

In [ ]:
def make_coverage(df):
    x = df.dropna(subset=["STATION_ID", "DATE", "VALUE_num"]).copy()
    x["YEAR_INT"] = x.DATE.dt.year
    cov = x.groupby(["source", "STATION_ID", "YEAR_INT"], as_index=False).agg(
        rows=("VALUE_num", "size"), unique_dates=("DATE", "nunique"),
        zero_count=("VALUE_num", lambda s: int((s == 0).sum())),
        annual_sum_observed=("VALUE_num", "sum"),
        first_date=("DATE", "min"), last_date=("DATE", "max"))
    cov["expected_days"] = cov.YEAR_INT.map(lambda y: 366 if pd.Timestamp(y, 12, 31).dayofyear == 366 else 365)
    cov["missing_calendar_days"] = cov.expected_days - cov.unique_dates
    cov["coverage_pct"] = 100 * cov.unique_dates / cov.expected_days
    return cov

coverage = pd.concat([make_coverage(station), make_coverage(history)], ignore_index=True)
annual_presence = coverage.groupby(["source", "YEAR_INT"]).agg(
    stations=("STATION_ID", "nunique"), station_year_pairs=("STATION_ID", "size"),
    median_coverage_pct=("coverage_pct", "median"),
    min_coverage_pct=("coverage_pct", "min"),
    pairs_below_90pct=("coverage_pct", lambda s: int((s < 90).sum()))).reset_index()
print("Resumen por año: estaciones con algún registro, cobertura y pares bajo 90%")
display(annual_presence)
for source, group in coverage.groupby("source"):
    years = set(group.YEAR_INT.astype(int))
    missing_years = sorted(set(range(min(years), max(years) + 1)) - years)
    print(source, "años con algún dato:", min(years), "a", max(years), "años vacíos dentro del rango:", missing_years)
print("Atención: la presencia de al menos una estación no significa que todas las estaciones tengan datos ese año.")
print("Estaciones-año con menos de 90% cobertura:", int((coverage.coverage_pct < 90).sum()))
display(coverage.sort_values("coverage_pct").head(15))

fig, ax = plt.subplots(figsize=(9, 4))
for source, group in coverage.groupby("source"):
    ax.hist(group.coverage_pct, bins=np.arange(0, 105, 5), alpha=.55, label=source)
ax.set(xlabel="Cobertura de fechas del año (%)", ylabel="Número de pares estación-año", title="Cobertura anual de estaciones")
ax.legend(); ax.grid(alpha=.2); plt.show()


## 5. Duplicados y fechas repetidas

Una fecha repetida puede ser duplicación accidental o varias mediciones para un mismo día. Hay que resolverlo antes de construir curvas anuales. La búsqueda se hace por fuente, estación y fecha.

In [ ]:
def find_duplicates(df):
    x = df.dropna(subset=['STATION_ID', 'DATE']).copy()
    counts = x.groupby(['source', 'STATION_ID', 'DATE']).size().rename('n_filas').reset_index()
    return counts[counts.n_filas > 1].sort_values('n_filas', ascending=False)

duplicates = find_duplicates(station)
print('Grupos de estación-fecha repetidos:', len(duplicates))
display(duplicates.head(20))

# Filas completamente idénticas (excluye las columnas auxiliares).
for name, df in frames.items():
    print(name, 'filas duplicadas idénticas:', int(df.duplicated().sum()))

## 6. Ceros, distribución y valores extremos

En lluvia, muchos ceros pueden ser días secos reales. Sin un código de calidad o manual de la fuente, no se pueden cambiar automáticamente a nulos. Los máximos extremos también pueden ser reales, acumulados por varios días o errores de digitación. Primero localizamos esos registros y los mostramos con estación y fecha.

In [ ]:
def value_summary(df, source):
    v = pd.to_numeric(df.VALUE, errors='coerce')
    return {'fuente': source, 'n': int(v.notna().sum()), 'ceros': int((v == 0).sum()),
            'ceros_pct': 100 * float((v == 0).mean()), 'negativos': int((v < 0).sum()),
            'media': v.mean(), 'mediana': v.median(), 'p99': v.quantile(.99),
            'p999': v.quantile(.999), 'maximo': v.max(), 'mayores_500': int((v > 500).sum())}

value_stats = pd.DataFrame([value_summary(df, name) for name, df in frames.items()] + [value_summary(hist, 'excel_1930_1979')])
display(value_stats.round(3))

def show_extremes(df, source, threshold=100):
    x = prepare_station(df, source)
    return x.loc[x.VALUE_num >= threshold, ['STATION_ID','YEAR','MONTH','DAY','DATE','VALUE']].sort_values('VALUE', ascending=False)

for name, df in list(frames.items()) + [('excel_1930_1979', hist)]:
    print('Valores >= 100:', name)
    display(show_extremes(df, name).head(20))

# Histograma separado: el pico en cero tapa la forma de los valores positivos.
positive = station.loc[station.VALUE_num > 0, 'VALUE_num'].dropna()
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(positive.clip(upper=positive.quantile(.99)), bins=60, color='teal', edgecolor='white')
ax.set(title='Valores positivos de lluvia (recortados visualmente al percentil 99)', xlabel='VALUE', ylabel='Frecuencia')
ax.grid(alpha=.2); plt.show()
# Revisión exploratoria de valores grandes justo al cierre del mes.
# Esto ayuda a detectar posibles acumulados mensuales digitados en un solo día.
for name, df in list(frames.items()) + [("excel_1930_1979", hist)]:
    x = prepare_station(df, name).dropna(subset=["DATE", "VALUE_num"]).copy()
    x["es_ultimo_dia_mes"] = x.DATE.dt.is_month_end
    hi = x[x.VALUE_num >= 100]
    if len(hi):
        print(name, "valores >=100:", len(hi), "en fin de mes:", int(hi.es_ultimo_dia_mes.sum()),
              "porcentaje fin de mes:", round(100 * hi.es_ultimo_dia_mes.mean(), 1))
        display(hi.sort_values("VALUE_num", ascending=False)[["STATION_ID","DATE","VALUE","es_ultimo_dia_mes"]].head(15))


## 7. Ceros sospechosos y patrones de digitación

Un indicio de cero usado como faltante puede ser una estación-año con proporción de ceros casi total o bloques de fechas idénticos a cero. Es solo una bandera: una zona seca también puede registrar muchos días secos. Buscamos además valores extremos que concentren una gran parte del total anual observado.

In [ ]:
zero_check = coverage.copy()
zero_check['zero_pct_rows'] = 100 * zero_check.zero_count / zero_check.rows
print('Estación-año con >= 95% filas cero (revisar, no borrar):', int((zero_check.zero_pct_rows >= 95).sum()))
display(zero_check.loc[zero_check.zero_pct_rows >= 95].sort_values('zero_pct_rows', ascending=False).head(20))

x = station.dropna(subset=['STATION_ID','DATE','VALUE_num']).copy()
x['year'] = x.DATE.dt.year
annual = x.groupby(['source','STATION_ID','year']).VALUE_num.sum().rename('annual_sum').reset_index()
x = x.merge(annual, on=['source','STATION_ID','year'], how='left')
x['daily_share_of_annual'] = np.where(x.annual_sum > 0, x.VALUE_num / x.annual_sum, np.nan)
suspect_days = x.loc[(x.VALUE_num >= 100) | (x.daily_share_of_annual >= .5),
                     ['source','STATION_ID','DATE','VALUE_num','annual_sum','daily_share_of_annual']]
display(suspect_days.sort_values('VALUE_num', ascending=False).head(30))

## 8. Periodos por fuente y posibilidad de emparejamiento

Los CSV de estaciones cubren 1980–2003 en conjunto; el archivo histórico llega hasta 1979. El satélite comienza en 1981 y llega a 2013, aunque 2013 tiene solo 24 archivos. El periodo común no se debe fijar solo por extremos de fechas: también depende de cobertura estación-año y de tener ubicación para asignar el píxel.

In [ ]:
periods = []
for name, df in list(frames.items()) + [('excel_1930_1979', hist)]:
    y = pd.to_numeric(df.YEAR, errors='coerce')
    periods.append({'fuente': name, 'filas': len(df), 'estaciones': df.STATION_ID.nunique(),
                    'primer_anio': int(y.min()), 'ultimo_anio': int(y.max())})
display(pd.DataFrame(periods))

all_ids = {name: set(df.STATION_ID.dropna().astype(str)) for name, df in list(frames.items()) + [('excel_1930_1979', hist)]}
print('IDs compartidos CSV 1980-90 y CSV 1991-2003:', len(all_ids['csv_1980_1990'] & all_ids['csv_1991_2003']))
print('Ojo: ID compartido no garantiza coordenadas iguales ni continuidad del sensor.')

## 9. Inventario de imágenes satelitales

Los nombres `pptYYYYNN.tif` parecen codificar año y periodo dentro del año. El inventario verifica cuántos TIFF existen por año y omite los archivos `._...` que macOS agrega como metadatos auxiliares. Este conteo no confirma unidades, resolución, proyección, nodata ni si los valores son acumulados pentadales.

In [ ]:
sat_zip = ROOT / 'ImagenesSatelitePentadal.zip'
with zipfile.ZipFile(sat_zip) as z:
    names = z.namelist()
    tiffs = [n for n in names if n.lower().endswith('.tif') and '/._' not in n]
    apple_meta = [n for n in names if '/._' in n]
    year_counts = Counter(Path(n).stem[3:7] for n in tiffs)
sat_year = pd.DataFrame([{'year': int(y), 'tiff_count': n, 'complete_72_expected': n == 72}
                         for y, n in sorted(year_counts.items())])
print('TIFF de datos:', len(tiffs), '| archivos AppleDouble:', len(apple_meta))
display(sat_year)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(sat_year.year, sat_year.tiff_count, color=np.where(sat_year.complete_72_expected, 'steelblue', 'darkorange'))
ax.axhline(72, color='black', linestyle='--', linewidth=1, label='72 periodos/año según patrón')
ax.set(title='Cantidad de TIFF por año', xlabel='Año', ylabel='Número de TIFF'); ax.legend(); ax.grid(axis='y', alpha=.2); plt.show()

# Intento opcional de leer metadatos de un TIFF. Si rasterio no está instalado, el conteo anterior sigue siendo válido.
try:
    import rasterio
    sample_path = '/tmp/eda_ppt_sample.tif'
    with zipfile.ZipFile(sat_zip) as z, open(sample_path, 'wb') as f:
        f.write(z.read(tiffs[0]))
    with rasterio.open(sample_path) as ds:
        print({'archivo': Path(tiffs[0]).name, 'ancho': ds.width, 'alto': ds.height,
               'bandas': ds.count, 'CRS': str(ds.crs), 'nodata': ds.nodata,
               'tipo': ds.dtypes, 'transform': ds.transform, 'bounds': ds.bounds})
        band = ds.read(1, masked=True)
        print('mínimo/máximo válidos:', float(band.min()), float(band.max()))
except ImportError:
    print('rasterio no está instalado: no se leyeron metadatos ni valores de píxel. Instálalo en tu entorno si el curso lo permite.')
except Exception as e:
    print('No se pudo leer el TIFF con rasterio:', repr(e))

## 10. Conclusiones iniciales y decisiones pendientes

### Lo que este EDA sí establece
- Hay una brecha temporal entre la serie histórica de estaciones (1930–1979), los CSV (1980–2003) y el satélite (1981–2013).
- La gran proporción de ceros hace necesario validar códigos de calidad y registros faltantes antes de interpretar lluvia seca.
- La cobertura debe evaluarse por estación-año, no solo por cantidad total de filas.
- Los extremos y días que concentran gran parte del total anual requieren investigación contextual.
- El satélite parece tener 72 periodos anuales en 1981–2012 y solo 24 en 2013.

### Lo que falta antes de modelar
1. Catálogo de estaciones con latitud, longitud, altitud, nombre y cambios de sensor.
2. Diccionario de `VALUE`, unidad, códigos de faltantes y reglas de acumulación de lluvia.
3. Metadatos del TIFF: CRS, resolución espacial, unidades, escala, nodata y significado de los 72 periodos.
4. Regla reproducible para cobertura mínima y tratamiento de duplicados, acumulados y ceros sospechosos.
5. Confirmar cómo evaluar el periodo 2004–2013, donde este inventario no encuentra CSV de estaciones.

**No elimines ni imputes datos por una regla automática basada solo en este EDA.** Registra la decisión y compara cuántas estaciones y pares estación-año quedan antes y después de cada filtro.